# SMS Spam Detection

This notebook documents the data science process for a simple spam detector using TF-IDF and Logistic Regression. The goal is to build a reproducible model that can be used by the Streamlit app without data leakage.

## 1. Problem Definition

The task consists of classifying SMS messages as `ham` or `spam`. Since spam is usually the minority class, accuracy alone is not enough. The evaluation focuses on spam precision, spam recall, spam F1, ROC-AUC, average precision, and the confusion matrix.

In [ ]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebook' else Path.cwd()
sys.path.append(str(PROJECT_ROOT))

import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

from app.model_utils import load_sms_dataset, train_model

DATA_PATH = PROJECT_ROOT / 'spam.csv'

## 2. Load and Clean Data

The original CSV has useful columns `v1` and `v2`, plus empty unnamed columns. The cleaning step keeps only the label and message, normalizes labels, removes empty messages, encodes the target, and removes duplicated SMS messages before splitting to reduce leakage risk.

In [ ]:
df = load_sms_dataset(DATA_PATH)
df.head()

In [ ]:
df.info()
df['label'].value_counts()

## 3. Exploratory Data Analysis

Useful EDA for this problem checks class balance and simple text-length differences. If spam is a minority class, stratified splitting and class weighting are appropriate.

In [ ]:
balance = df['label'].value_counts().to_frame('count')
balance['percent'] = (balance['count'] / len(df) * 100).round(2)
balance

In [ ]:
df['message_length'] = df['message'].str.len()
df.groupby('label')['message_length'].describe().round(2)

In [ ]:
sns.countplot(data=df, x='label')
plt.title('Class distribution')
plt.show()

sns.boxplot(data=df, x='label', y='message_length')
plt.title('Message length by class')
plt.show()

## 4. Modeling Strategy

To avoid data leakage, the train/test split is done before fitting TF-IDF. TF-IDF and Logistic Regression are wrapped in one `Pipeline`, so vocabulary learning happens only on training folds. `GridSearchCV` is applied only on the training data with stratified folds. To reduce overfitting, the model uses TF-IDF document-frequency limits, n-gram validation, regularized Logistic Regression, and cross-validation.

In [ ]:
metrics = train_model()
metrics['best_params'], metrics['cv_best_f1']

## 5. Final Test Metrics

The final metrics below are calculated on the holdout test set, which was not used to fit TF-IDF or choose hyperparameters.

In [ ]:
metrics['test_metrics']

In [ ]:
metrics['confusion_matrix']

## 6. Inference

The Streamlit app loads the persisted pipeline from `models/spam_model.joblib`, so inference uses the same trained TF-IDF vocabulary and Logistic Regression model.